# OpenAI API 종합 복습: 음성 FAQ 상담 도우미

음성 FAQ 상담 도우미는 사용자의 음성 질문을 텍스트로 바꾸고, 안전성을 확인한 뒤, 관련 FAQ를 찾아 답변과 안내 음성을 만드는 작은 애플리케이션이다. 하나의 API 기능만 확인하는 예제가 아니라 여러 API의 출력이 다음 API의 입력으로 이어지는 파이프라인을 구현한다.

이 실습이 필요한 이유는 실제 서비스가 하나의 모델 호출로 완성되지 않기 때문이다. 음성 인식, 입력 검사, 의미 검색, 답변 생성, 음성 합성을 순서대로 연결하면 각 단계의 책임과 데이터 형태를 함께 복습할 수 있다.



## 문제: 음성 FAQ 상담 도우미 구현하기

사용자가 음성으로 남긴 교육 과정 문의를 처리하는 프로그램을 구현한다. 수업에서는 마이크 녹음 대신 TTS로 테스트 음성을 먼저 만들며, 실제 녹음 파일이 있다면 같은 위치에 넣어 바꿔 사용할 수 있다.

처리 순서는 다음과 같다.

`질문 텍스트 → TTS 테스트 음성 → STT 전사문 → Moderation 검사 → FAQ 의미 검색 → 답변 생성 → TTS 안내 음성`

### 필수 요구 사항

- `.env`의 `OPENAI_API_KEY`를 이용해 API 클라이언트를 만든다.
- 테스트 질문을 MP3 파일로 생성한다.
- 생성한 MP3를 전사하고 전사문을 출력한다.
- Moderation의 `flagged`와 True인 범주를 확인한다.
- 질문과 FAQ를 같은 임베딩 모델로 변환한다.
- Cosine Similarity가 가장 높은 FAQ 한 건을 선택한다.
- 모델이 검색된 FAQ 범위 안에서만 답변하도록 지시한다.
- 최종 답변을 MP3로 저장하고 재생한다.

### 완료 결과

실행이 끝나면 전사문, Moderation 결과, 검색된 FAQ와 유사도, 최종 답변, 답변 음성 파일을 확인할 수 있어야 한다. API 요청은 비용이 발생하므로 같은 셀을 불필요하게 반복 실행하지 않는다. 생성 음성을 사용자에게 제공할 때는 AI가 만든 음성임을 안내한다.


## 파이프라인에서 사용하는 API

각 API는 서로 다른 형태의 값을 반환한다. 파일 경로, 문자열, Boolean, 숫자 벡터를 구분해야 다음 단계에 올바르게 전달할 수 있다.

- **Speech API**: 문자열을 MP3 음성 파일로 변환한다.
- **Transcription API**: MP3 파일을 전사문 문자열로 변환한다.
- **Moderation API**: 입력에서 검토할 신호가 있는지 Boolean과 범주별 점수로 반환한다.
- **Embeddings API**: 문장을 의미를 나타내는 숫자 벡터로 변환한다.
- **Cosine Similarity**: 두 벡터의 방향이 얼마나 비슷한지 계산한다. 1에 가까울수록 방향이 비슷하지만, 모든 데이터에 공통으로 적용되는 정답 임계값을 뜻하지 않는다.
- **Responses API**: 질문과 검색 근거를 입력받아 사용자에게 보여 줄 답변 문자열을 생성한다.

공식 문서는 다음과 같다.

- [Text-to-Speech 가이드](https://developers.openai.com/api/docs/guides/text-to-speech)
- [Speech-to-Text 가이드](https://developers.openai.com/api/docs/guides/speech-to-text)
- [Moderation 가이드](https://developers.openai.com/api/docs/guides/moderation)
- [Embeddings 가이드](https://developers.openai.com/api/docs/guides/embeddings)
- [Text generation과 Responses API](https://developers.openai.com/api/docs/guides/text)


### API 클라이언트와 결과 폴더 준비

`.env` 설정은 앞 단원에서 완료했다고 가정한다. `find_dotenv()`는 현재 작업 폴더부터 상위 폴더로 이동하며 `.env`를 찾고, `OpenAI()`는 환경 변수의 API 키를 사용한다. 키 값은 출력하지 않는다.


In [17]:
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import Audio, display
from openai import OpenAI
from unicodedata import category

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위의 .env 파일을 확인한다.")
load_dotenv(dotenv_path, override=False)

client = OpenAI()

output_dir = Path("review_outputs")
output_dir.mkdir(exist_ok=True)

print("결과 폴더:", output_dir)

결과 폴더: review_outputs


### 테스트 질문과 FAQ 데이터 준비

FAQ(Frequently Asked Questions)는 자주 묻는 질문과 답변을 모은 데이터이다. 검색할 때는 질문만 임베딩하지 않고 질문과 답변을 하나의 문서 문자열로 결합한다. 그러면 사용자의 표현이 FAQ 제목과 다르더라도 답변 본문의 의미를 함께 비교할 수 있다.

각 FAQ는 `question`과 `answer`를 가진 딕셔너리이다. 목록의 순서는 이후 임베딩 벡터의 순서와 같아야 검색된 벡터를 다시 FAQ 문장으로 연결할 수 있다.


In [18]:
# sample_question과 question·answer 딕셔너리 5개를 준비한다.
# 각 FAQ의 질문과 답변을 하나의 검색 문서 문자열로 결합한다.
# 문서 목록과 FAQ 목록의 순서를 동일하게 유지한다.
sample_question = "개인 사정으로 결석하면 출석으로 인정받을 수 있나요?"

faq_items = [
    {
        "question": "결석한 날을 출석으로 인정받을 수 있나요?",
        "answer": "공식 증빙이 있는 사유는 운영 규정에 따라 출석 인정 여부를 확인한다. 증빙 서류를 담당자에게 제출해야 한다.",
    },
    {
        "question": "과제는 어디에 제출하나요?",
        "answer": "과제는 안내된 학습 관리 시스템의 해당 주차 과제 메뉴에 제출한다.",
    },
    {
        "question": "수업 자료는 어디에서 받나요?",
        "answer": "수업 자료는 과정 공유 저장소에서 내려받으며 수업 전 공지된 폴더를 확인한다.",
    },
    {
        "question": "OpenAI API 사용 비용은 누가 부담하나요?",
        "answer": "API 비용 정책은 과정 공지를 따르며 실습 전 사용 한도와 결제 설정을 확인한다.",
    },
    {
        "question": "수료하려면 어떤 조건을 충족해야 하나요?",
        "answer": "수료 조건은 출석률과 필수 과제 및 프로젝트 기준으로 판단하며 세부 기준은 과정 운영 규정을 확인한다.",
    },
]

# 질문과 답변 사이에 줄바꿈을 넣어 하나의 검색 문서로 만들고 FAQ 순서를 유지한다.
faq_documents = [
    f"질문: {item['question']}\n답변: {item['answer']}"
    for item in faq_items
]

print("테스트 질문:", sample_question)
print("FAQ 개수:", len(faq_documents))


테스트 질문: 개인 사정으로 결석하면 출석으로 인정받을 수 있나요?
FAQ 개수: 5


## 1단계: TTS로 테스트용 음성 질문 만들기

마이크 녹음 환경은 운영체제와 장치에 따라 달라질 수 있다. 이 실습에서는 동일한 입력으로 파이프라인을 비교하기 위해 질문 문자열을 TTS로 변환한다. 생성된 `review_question.mp3`는 다음 STT 셀의 파일 입력이 된다.

- `model='tts-1'`: 응답 속도를 우선하는 음성 생성 모델이다.
- `voice='nova'`: 생성할 목소리를 선택한다.
- `response_format='mp3'`: 저장할 음성 파일 형식을 지정한다.
- `stream_to_file()`: 반환된 음성 바이트를 지정한 경로에 저장한다.


In [19]:
# Speech API에 sample_question과 tts-1, nova, mp3 설정을 전달한다.
# 응답을 review_outputs/review_question.mp3에 저장하고 Audio로 재생한다.

question_audio_path = output_dir / "review_question.mp3"

with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="nova",
    input=sample_question,
    response_format="mp3"
) as speech_response:
    speech_response.stream_to_file(question_audio_path)

print("질문: ", sample_question)
display(Audio(question_audio_path))

질문:  개인 사정으로 결석하면 출석으로 인정받을 수 있나요?


## 2단계: STT로 음성 질문 전사하기

STT는 음성 파일의 소리를 전사문 문자열로 바꾼다. 파일은 바이너리 읽기 모드인 `rb`로 열어야 하며, 반환된 `transcription.text`가 Moderation과 의미 검색의 공통 입력이 된다.

실제 녹음 파일을 사용할 때는 `question_audio_path`만 해당 MP3 또는 WAV 상대 경로로 바꾼다. 전사 결과에 오탈자가 있으면 이후 검색 결과도 달라질 수 있으므로 반드시 전사문을 먼저 확인한다.


In [20]:
# 음성 파일을 rb 모드로 열고 gpt-transcribe에 전달한다.
# prompt로 한국어 교육 과정 문의 문맥을 주고 transcription.txt를 정리한다.
with question_audio_path.open("rb") as audio_file:

    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        prompt="한국어 교육 과정 문의이다. 출결, 과제, 수업 자료, API 비용, 수료 조건에 관한 질문이다.",
    )

transcribed_question = transcription.text.strip()
print("전사문:", transcribed_question)

전사문: 개인 사정으로 결석하면 출석으로 인정받을 수 있나요?


## 3단계: Moderation으로 입력 신호 확인하기

Moderation은 전사문에서 검토가 필요한 콘텐츠 신호를 확인한다. `flagged`는 전체 판정이고 `categories`는 범주별 Boolean이다. 이 실습에서는 `flagged=True`이면 자동 답변을 만들지 않고 사람 검토 대상으로 보내는 간단한 수업용 정책을 적용한다.

이 정책은 모든 서비스의 공통 규칙이 아니다. 실제 서비스에서는 사용자 맥락, 법적 요구, 사람 검토 절차를 함께 고려하여 허용·검토·지원 안내 기준을 별도로 정해야 한다.


In [21]:
# omni-moderation-latest로 전사문을 검사한다.
# flagged와 True 범주를 확인하고 allow일 때만 검색 단계로 진행한다.
moderation_response = client.moderations.create(
    model="omni-moderation-latest",
    input=transcribed_question,
)

moderation_result = moderation_response.results[0]

CATEGORY_ALIASES = {
    "harassment_threatening": "harassment/threatening",
    "hate_threatening": "hate/threatening",
    "illicit_violent": "illicit/violent",
    "self_harm": "self-harm",
    "self_harm_intent": "self-harm/intent",
    "self_harm_instructions": "self-harm/instructions",
    "sexual_minors": "sexual/minors",
    "violence_graphic": "violence/graphic",
}

category_flags = moderation_result.categories.model_dump()

flagged_categories = [
    CATEGORY_ALIASES.get(category, category)
    for category, is_flagged in category_flags.items()
    if is_flagged
]


# 목표: 전사문의 전체 moderation 판정과 True인 범주를 확인한다.
moderation_response = client.moderations.create(
    model="omni-moderation-latest",
    input=transcribed_question,
)
moderation_result = moderation_response.results[0]

# 중첩 범주는 Python 속성명과 API 표기가 다르므로 이전 Moderation 단원과 같은 대응표를 사용한다.
category_aliases = {
    "harassment_threatening": "harassment/threatening",
    "hate_threatening": "hate/threatening",
    "illicit_violent": "illicit/violent",
    "self_harm": "self-harm",
    "self_harm_intent": "self-harm/intent",
    "self_harm_instructions": "self-harm/instructions",
    "sexual_minors": "sexual/minors",
    "violence_graphic": "violence/graphic",
}
category_flags = moderation_result.categories.model_dump()
flagged_categories = [
    category_aliases.get(category, category)
    for category, is_flagged in category_flags.items()
    if is_flagged
]

# 전체 판정은 다음 셀에서 검색을 계속할지 결정하는 수업용 정책 입력이 된다.
policy_action = "human_review" if moderation_result.flagged else "allow"
print("전체 판정:", moderation_result.flagged)
print("True 범주:", flagged_categories)
print("처리 경로:", policy_action)

if policy_action != "allow":
    raise RuntimeError("검토가 필요한 입력이므로 자동 답변 생성을 중단한다.")

전체 판정: False
True 범주: []
처리 경로: allow


## 4단계: Embeddings로 관련 FAQ 검색하기

임베딩은 문장을 고정 길이 숫자 벡터로 표현한다. 사용자 질문과 FAQ 문서를 같은 모델로 임베딩하면 문장에 사용된 단어가 정확히 같지 않아도 의미가 가까운 문서를 찾을 수 있다.

Cosine Similarity는 두 벡터 사이의 각도를 이용한다. 벡터를 각각 길이 1로 정규화한 뒤 내적하면 코사인 유사도가 되며, 이 실습에서는 점수가 가장 큰 FAQ 한 건을 선택한다. 높은 점수가 반드시 정답을 뜻하지 않으므로 최종 결과에서 질문과 검색 문서가 실제로 관련 있는지도 확인해야 한다.

API 요청 횟수를 줄이기 위해 사용자 질문과 FAQ 5건을 하나의 목록으로 묶어 한 번에 임베딩한다. 응답의 첫 벡터는 질문이고 나머지 벡터는 FAQ 목록과 같은 순서이다.


In [22]:
# 질문과 FAQ 문서를 text-embedding-3-small로 한 번에 임베딩한다.
# 각 벡터를 정규화하고 질문 벡터와 FAQ 벡터의 코사인 유사도를 계산한다.
# np.argmax로 가장 높은 FAQ의 인덱스와 점수를 찾는다.
import numpy as np

embedding_inputs = [transcribed_question, *faq_documents]

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=embedding_inputs,
)

# 응답 순서를 유지한 벡터 목록
embedding_matrix = np.asarray(
    [item.embedding for item in embedding_response.data],
    dtype=np.float32,
)

# 각 행 벡터를 길이 1로 정규화하면 두 행의 내적이 코사인 유사도가 된다.
row_norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
normalized_matrix = embedding_matrix / np.clip(row_norms, 1e-12, None)


# 첫 행은 질문 벡터이고 1번 이후 행은 FAQ 벡터이다. 결과는 FAQ 개수 5와 같은 길이이다.
similarity_scores = normalized_matrix[1:] @ normalized_matrix[0]
best_index = int(np.argmax(similarity_scores))
best_faq = faq_items[best_index]
best_score = float(similarity_scores[best_index])

print("검색된 질문:", best_faq["question"])
print("검색된 답변:", best_faq["answer"])
print("코사인 유사도:", round(best_score, 4))

검색된 질문: 결석한 날을 출석으로 인정받을 수 있나요?
검색된 답변: 공식 증빙이 있는 사유는 운영 규정에 따라 출석 인정 여부를 확인한다. 증빙 서류를 담당자에게 제출해야 한다.
코사인 유사도: 0.6001


## 5단계: 검색 근거로 최종 답변 생성하기

검색 증강 답변은 모델의 일반 지식만 사용하는 대신 검색된 문서를 프롬프트에 함께 제공한다. 여기서는 하나의 FAQ만 근거로 사용하며, 근거에 없는 내용을 추측하지 말고 담당자 확인이 필요하다고 답하도록 제한한다.

- `instructions`: 상담 도우미의 공통 역할과 답변 규칙을 전달한다.
- `input`: 이번 질문과 검색된 FAQ 근거를 전달한다.
- `response.output_text`: TTS 입력으로 사용할 최종 답변 문자열이다.


In [23]:
# 사용자 질문과 선택한 FAQ를 하나의 입력 문자열로 구성한다.
# Responses API에 근거 제한 instructions와 입력을 전달한다.
# response.output_text를 최종 답변 문자열로 저장한다.
answer_input = f"""
사용자 질문:
{transcribed_question}

검색된 FAQ:
질문: {best_faq['question']}
답변: {best_faq['answer']}
""".strip()

answer_response = client.responses.create(
    model="gpt-5.6-luna",
    reasoning={"effort": "none"},
    instructions=(
        "교육 과정 FAQ 상담 도우미이다. 제공된 FAQ만 근거로 두세 문장으로 답한다. "
        "FAQ에 없는 내용은 추측하지 말고 담당자 확인이 필요하다고 안내한다."
    ),
    input=answer_input,
)

# answer_text는 화면에 표시한 뒤 마지막 TTS 단계의 input으로 전달한다.
answer_text = answer_response.output_text.strip()
print("최종 답변:", answer_text)

최종 답변: 공식 증빙이 있는 사유라면 운영 규정에 따라 출석 인정 여부를 확인할 수 있습니다. 증빙 서류를 담당자에게 제출해 확인받으시기 바랍니다.


## 6단계: 최종 답변을 음성으로 제공하기

최종 답변 문자열을 다시 Speech API에 전달하면 음성 안내 파일을 만들 수 있다. 처음 만든 질문 음성과 파일명을 분리하여 입력과 출력이 뒤섞이지 않게 한다.

이 단계의 핵심은 TTS를 두 번 사용했다는 사실이 아니다. 첫 TTS는 재현 가능한 테스트 입력을 준비하고, 마지막 TTS는 파이프라인의 최종 사용자 출력을 만든다.


In [24]:
# Speech API에 최종 답변을 전달해 review_answer.mp3로 저장한다.
# 질문 음성과 다른 파일명을 사용하고 Audio로 재생한다.

answer_audio_path = output_dir / "review_answer.mp3"

with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="nova",
    input=answer_text,
    response_format="mp3",
) as speech_response:
    speech_response.stream_to_file(answer_audio_path)

display(Audio(answer_audio_path))
